In [ ]:
import numpy as np

from typing import Union, List
from pymatching import Matching

In [ ]:


base_relations = {
    ('X', 'Z'): -1,
}

def c(a, b):
    pass

def prod(A, B):
    pass

In [3]:
print(X(5, [1, 3]))

['I', 'X', 'I', 'X', 'I']


In [4]:
def detect_changes(syndrome: list) -> list:
    syndrome = np.array(syndrome)

    syndrome_change = np.abs(np.diff(syndrome, axis=0, prepend=np.zeros((1, syndrome.shape[1]))))

    return syndrome_change


def build_matching_graph(rounds, px, pm):
    matching = Matching()

    if px > 0:
        weightx = -np.log(px/(1-px))
    if pm > 0:
        weightm = -np.log(pm/(1-pm))


    for i in range(rounds):
        # Adding physical edges
        if px > 0:
            matching.add_boundary_edge(2*i, weight=weightx)
            matching.add_edge(2*i, 2*i+1, weight=weightx)
            matching.add_boundary_edge(2*i+1, weight=weightx)

        # Adding measurement edges (currently no time boundry TODO)
        if pm > 0:
            if i < rounds - 1:
                matching.add_edge(2*i, 2*(i + 1), weight=weightm)
                matching.add_edge(2*i + 1, 2*(i + 1) + 1, weight=weightm)

    return matching


def match_errors(matching_graph: Matching, detection_events) -> list:

    matched = matching_graph.decode_to_matched_dets_array(detection_events.flatten())
    
    return matched


def extract_predicted_errors(matched_errors):
    errors = np.zeros(3)
    for match in matched_errors:
        # TODO. This works specifically for 3 bit repitition code with bitflip noise
        if -1 not in match:
            s1, t1 = match[0] % 2, match[0] // 2
            s2, t2 = match[1] % 2, match[1] // 2

            if s1 - s2 != 0:
                errors[1] += 1
        else:
            for i in match:
                if i != -1:
                    s = i % 2

                    if s == 0:
                        errors[0] += 1
                    else:
                        errors[2] += 1

    return errors % 2

In [ ]:

rels = {
    ('X', 'Z'): '-Y',
    ('Z', 'X'): 'Y',
    ('X', 'Y'): 'Z',
    ('Y', 'X'): '-Z',
    ('Y', 'Z'): 'X',
    ('Z', 'Y'): '-X',
}

# TODO: Chack assumptions
def get_syndrome_bit(state, stabilizer):
    bit = 0
    for comp in zip(state, stabilizer):
        if 'I' not in comp:
            if comp[0] != comp[1]:
                bit += 1

    return bit % 2


rounds = 20
shots = 20000
px = 0.04
pm = 0.01

intial_state = Operator(3)

S1 = Z(3, [0, 1])
S2 = Z(3, [1, 2])

# stabs = [0, 0]
matching_graph = build_matching_graph(rounds, px, pm)
# er = {}


result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = [0, 0, 0]
    state = intial_state.copy()
    for rnd in range(rounds):

        
        for i in range(3):
            if np.random.random() < px:
                state = state * X(3, i)
                true_phys_error[i] += 1

        sb1 = get_syndrome_bit(state, S1)
        sb2 = get_syndrome_bit(state, S2)

        syndrome.append([sb1, sb2])

    
    detections = detect_changes(np.array(syndrome))
    # print(detections)
    matches = match_errors(matching_graph, detections)
    predicted_errors = extract_predicted_errors(matches)

    true_total_error = [i % 2 for i in true_phys_error]
    
    diff = sum(np.abs(true_total_error - predicted_errors))
    
    if diff != 0:
        result += 1

print(result/shots)
        

0.08525
